# PR_DSS_Reachability — procedure-forward reachability over cosmos-graph

Produces `consumer-bases/interim/PR_DSS_Reachability.xlsx` with three sheets:

| Sheet | Grain | Source |
|---|---|---|
| `ReadMe` | — | provenance and column dictionary |
| `PR_Procedures` | one row per governed PROCEDUR term (152 rows) | aggregation over PR-domain Variables PRDECOD/PRLOC slots |
| `PR_Reachability` | one row per (PROCEDUR, LOC) addressed | combinations that any PR DSS pins or value-list-includes on both axes |

## Why

Counterpart on the procedure side to what `Specimen_Findings`, `Measurement_Findings`, and `Instrument_Findings` (in `sdtm-findings-graph/`) do on the Findings side: pre-join the standard's implicit relationships into traversable rows.

Closes the asymmetry between PR-side and Findings-side coverage. Surfaces the inventory's universal finding ("none of 21 procedures has a PR DSS at right grain") as queryable rows rather than something to re-derive per inventory run.

## Inputs

| File | Track | Sheets used |
|---|---|---|
| `cosmos-graph/interim/COSMoS_Graph.xlsx` | cosmos-graph | DSS, Variables |
| `cosmos-graph/interim/COSMoS_Graph_CT.xlsx` | cosmos-graph | CodelistTerms |

## Output

`consumer-bases/interim/PR_DSS_Reachability.xlsx`

## Scope discipline

All classifications are mechanical (deterministic from Variables-sheet state). No editorial judgement. Stays within `consumer-bases` scope.

## 1. Setup

In [1]:
import pandas as pd
import re
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

In [2]:
BASE_DIR = Path.cwd().parent          # consumer-bases/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/

GRAPH_FILE = REPO_ROOT / 'cosmos-graph' / 'interim' / 'COSMoS_Graph.xlsx'
GRAPH_CT_FILE = REPO_ROOT / 'cosmos-graph' / 'interim' / 'COSMoS_Graph_CT.xlsx'

INTERIM_DIR = BASE_DIR / 'interim'
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = INTERIM_DIR / 'PR_DSS_Reachability.xlsx'

PROCEDUR_CL = 'C101858'  # PROCEDUR codelist
LOC_CL = 'C74456'        # LOC codelist

for f, label in [(GRAPH_FILE, 'Graph'), (GRAPH_CT_FILE, 'Graph CT')]:
    if f.exists():
        print(f'  {label}: {f.relative_to(REPO_ROOT)}')
    else:
        raise FileNotFoundError(f'{label} file not found: {f}')

print(f'  Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')

  Graph: cosmos-graph/interim/COSMoS_Graph.xlsx
  Graph CT: cosmos-graph/interim/COSMoS_Graph_CT.xlsx
  Output: consumer-bases/interim/PR_DSS_Reachability.xlsx


## 2. Load inputs

In [3]:
vars_g = pd.read_excel(GRAPH_FILE, sheet_name='Variables', dtype=str).fillna('')
dss_g = pd.read_excel(GRAPH_FILE, sheet_name='DSS', dtype=str).fillna('')
codelist_terms = pd.read_excel(GRAPH_CT_FILE, sheet_name='CodelistTerms', dtype=str).fillna('')

print(f'Variables:      {len(vars_g):>6,} rows')
print(f'DSS:            {len(dss_g):>6,} rows')
print(f'CodelistTerms:  {len(codelist_terms):>6,} rows')

Variables:      12,677 rows
DSS:             1,326 rows
CodelistTerms:  17,523 rows


## 3. PR-domain slot extraction

Build one struct per PR DSS recording the state of its `PRDECOD` (procedure) and `PRLOC` (anatomy) slots — pinned, value_list-restricted, or bare.

In [4]:
# PR DSSs
pr_dss = dss_g[dss_g['domain'] == 'PR'][['ds_id', 'ds_short_name']].reset_index(drop=True)
print(f'PR-domain DSSs: {len(pr_dss)}')
print(pr_dss.to_string(index=False))

PR-domain DSSs: 8
                  ds_id                   ds_short_name
                 CTSCAN                         CT Scan
            CTSCANCHEST                   Chest CT Scan
                    MRI                             MRI
               MRIBRAIN                       MRI Brain
        RADIATIONCANCER        Radiation Therapy Cancer
RADTHERAPHYBREASTCANCER Radiation Therapy Breast Cancer
                   XRAY                           X-Ray
              XRAYCHEST                     Chest X-Ray


In [5]:
# PRDECOD / PRLOC variable rows in PR DSSs
pr_var_rows = vars_g[
    vars_g['ds_id'].isin(pr_dss['ds_id']) &
    vars_g['variable_name'].isin(['PRDECOD', 'PRLOC'])
].copy()

# Sanity: every PR DSS must have exactly one PRDECOD row and one PRLOC row
g = pr_var_rows.groupby(['ds_id', 'variable_name']).size().unstack(fill_value=0)
expected = pd.DataFrame(index=pr_dss['ds_id'], columns=['PRDECOD', 'PRLOC']).fillna(1)
missing = (g.reindex(pr_dss['ds_id']).fillna(0) != 1).any(axis=1)
if missing.any():
    raise RuntimeError(f'PR DSSs without exactly one PRDECOD+PRLOC row: {missing[missing].index.tolist()}')
print(f'PR variable rows extracted: {len(pr_var_rows)} (expected {2 * len(pr_dss)})')

PR variable rows extracted: 16 (expected 16)


/tmp/ipykernel_6/4070085934.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  expected = pd.DataFrame(index=pr_dss['ds_id'], columns=['PRDECOD', 'PRLOC']).fillna(1)


## 4. Parse value_list strings into term concept IDs

COSMoS source uses both `;` and `,` as separators in value_list strings, sometimes within the same domain. Some governed terms (e.g. PROCEDUR `ABLATION, ACCESSORY PATHWAYS`, LOC `ABDOMINAL QUADRANT, LEFT LOWER`) contain commas inside the submission value, so a naive comma split breaks those.

Strategy: try `;` split first. If all chunks match a governed term in the codelist, accept. Otherwise try `,\s*` split. Otherwise fall back to either-separator split and report unmatched chunks.

In [6]:
def _parse_value_list(s: str, governed_subs: set, governed_id_lookup: dict):
    """Return list of (term_concept_id, term_submission_value) for chunks matching governed terms.
    Also return list of unmatched chunks."""
    if not s:
        return [], []
    for sep in (';', ','):
        parts = [p.strip() for p in re.split(rf'{sep}\s*', s) if p.strip()]
        if parts and all(p in governed_subs for p in parts):
            return [(governed_id_lookup[p], p) for p in parts], []
    # Fallback: try either separator and report mismatches
    parts = [p.strip() for p in re.split(r'[;,]\s*', s) if p.strip()]
    matched = [(governed_id_lookup[p], p) for p in parts if p in governed_subs]
    unmatched = [p for p in parts if p not in governed_subs]
    return matched, unmatched


# Build governed-term lookups
proc_terms = codelist_terms[codelist_terms['codelist_concept_id'] == PROCEDUR_CL]
loc_terms  = codelist_terms[codelist_terms['codelist_concept_id'] == LOC_CL]

PROC_SUBS = set(proc_terms['term_submission_value'])
LOC_SUBS  = set(loc_terms['term_submission_value'])
PROC_ID   = dict(zip(proc_terms['term_submission_value'], proc_terms['term_concept_id']))
LOC_ID    = dict(zip(loc_terms['term_submission_value'],  loc_terms['term_concept_id']))

print(f'PROCEDUR governed terms: {len(PROC_SUBS)}')
print(f'LOC governed terms:      {len(LOC_SUBS)}')

PROCEDUR governed terms: 152
LOC governed terms:      1474


In [7]:
# Build per-DSS slot record
records = []
unmatched_log = []

for _, ds_row in pr_dss.iterrows():
    ds_id = ds_row['ds_id']
    prdecod = pr_var_rows[(pr_var_rows['ds_id'] == ds_id) & (pr_var_rows['variable_name'] == 'PRDECOD')].iloc[0]
    prloc   = pr_var_rows[(pr_var_rows['ds_id'] == ds_id) & (pr_var_rows['variable_name'] == 'PRLOC')].iloc[0]

    proc_pinned_id   = prdecod['assigned_term_concept_id'] or ''
    proc_pinned_sub  = prdecod['assigned_term_value'] or ''
    proc_vlist_pairs, proc_vlist_unmatched = _parse_value_list(prdecod['value_list'], PROC_SUBS, PROC_ID)
    proc_bare = (proc_pinned_id == '') and (not proc_vlist_pairs) and (prdecod['value_list'] == '')
    if proc_vlist_unmatched:
        unmatched_log.append((ds_id, 'PRDECOD', proc_vlist_unmatched))

    anat_pinned_id   = prloc['assigned_term_concept_id'] or ''
    anat_pinned_sub  = prloc['assigned_term_value'] or ''
    anat_vlist_pairs, anat_vlist_unmatched = _parse_value_list(prloc['value_list'], LOC_SUBS, LOC_ID)
    anat_bare = (anat_pinned_id == '') and (not anat_vlist_pairs) and (prloc['value_list'] == '')
    if anat_vlist_unmatched:
        unmatched_log.append((ds_id, 'PRLOC', anat_vlist_unmatched))

    records.append({
        'ds_id': ds_id,
        'ds_short_name': ds_row['ds_short_name'],
        'proc_pinned_id': proc_pinned_id,
        'proc_pinned_sub': proc_pinned_sub,
        'proc_vlist': proc_vlist_pairs,
        'proc_bare': proc_bare,
        'anat_pinned_id': anat_pinned_id,
        'anat_pinned_sub': anat_pinned_sub,
        'anat_vlist': anat_vlist_pairs,
        'anat_bare': anat_bare,
    })

print(f'PR slot records built: {len(records)}')
if unmatched_log:
    print('Unmatched value_list chunks:')
    for ds_id, var, chunks in unmatched_log:
        print(f'  {ds_id} {var}: {chunks}')
else:
    print('All value_list chunks matched governed terms.')

PR slot records built: 8
Unmatched value_list chunks:
  RADTHERAPHYBREASTCANCER PRDECOD: ['INTENSITY MODULATED RADIATION THERAPY', 'RADIOSURGERY', 'STEREOTACTIC BODY RADIATION THERAPY', 'INTRACAVITY BRACHYTHERAPY', 'INTERSTITIAL BRACHYTHERAPY']


## 5. PR_Procedures sheet — one row per governed PROCEDUR term

Procedure-forward view: for each of the 152 governed PROCEDUR terms, record how many PR DSSs reach it and at what grain.

In [8]:
rows = []
for _, t in proc_terms.iterrows():
    pid = t['term_concept_id']
    psub = t['term_submission_value']
    pinning = [r for r in records if r['proc_pinned_id'] == pid]
    in_vlist = [r for r in records if pid in [v[0] for v in r['proc_vlist']]]

    # For each pinning DSS, classify the PRLOC slot
    right_grain = [r for r in pinning if r['anat_pinned_id']]
    modality_vlist = [r for r in pinning if r['anat_vlist']]
    modality_bare = [r for r in pinning if r['anat_bare'] or (not r['anat_pinned_id'] and not r['anat_vlist'])]
    # Note: bare and 'no pin and empty value_list' overlap; modality_bare counts both

    if pinning:
        if right_grain:
            classification = 'right_grain_exists'
        else:
            classification = 'modality_only'
    else:
        classification = 'no_pr_dss'

    rows.append({
        'procedure_concept_id': pid,
        'procedure_submission_value': psub,
        'dss_count_pinning': len(pinning),
        'dss_count_in_value_list': len(in_vlist),
        'dss_count_at_right_grain': len(right_grain),
        'dss_count_modality_only_anatomy_vlist': len(modality_vlist),
        'dss_count_modality_only_anatomy_bare': len([r for r in pinning if r['anat_bare']]),
        'dss_ids_pinning': '; '.join(sorted(r['ds_id'] for r in pinning)),
        'dss_ids_in_value_list': '; '.join(sorted(r['ds_id'] for r in in_vlist)),
        'coverage_classification': classification,
    })

pr_procedures_out = pd.DataFrame(rows).sort_values('procedure_submission_value').reset_index(drop=True)
print(f'PR_Procedures: {len(pr_procedures_out):,} rows x {len(pr_procedures_out.columns)} cols')
print()
print('Coverage classification distribution:')
print(pr_procedures_out['coverage_classification'].value_counts())

PR_Procedures: 152 rows x 10 cols

Coverage classification distribution:
coverage_classification
no_pr_dss             149
right_grain_exists      3
Name: count, dtype: int64


## 6. PR_Reachability sheet — one row per addressed (PROCEDUR, LOC)

Universe = (procedure_term, anatomy_term) combinations that any PR DSS addresses on both axes — pinned-pinned, pinned-vlist, vlist-pinned, vlist-vlist. Combinations not in this set are absent from the artefact (the gap is the absence of a row, plus the `no_pr_dss` rows in PR_Procedures).

In [9]:
reach_rows = []

for r in records:
    # Procedure side: pinned procedure plus any procedures in value_list
    proc_pairs = []
    if r['proc_pinned_id']:
        proc_pairs.append(('pinned', r['proc_pinned_id'], r['proc_pinned_sub']))
    for pid, psub in r['proc_vlist']:
        proc_pairs.append(('vlist', pid, psub))

    # Anatomy side: pinned anatomy plus any anatomies in value_list
    anat_pairs = []
    if r['anat_pinned_id']:
        anat_pairs.append(('pinned', r['anat_pinned_id'], r['anat_pinned_sub']))
    for aid, asub in r['anat_vlist']:
        anat_pairs.append(('vlist', aid, asub))

    # Cross-product within this DSS
    for p_state, pid, psub in proc_pairs:
        for a_state, aid, asub in anat_pairs:
            reach_rows.append({
                'procedure_concept_id': pid,
                'procedure_submission_value': psub,
                'anatomy_concept_id': aid,
                'anatomy_submission_value': asub,
                'ds_id': r['ds_id'],
                'proc_state': p_state,    # pinned / vlist
                'anat_state': a_state,    # pinned / vlist
            })

raw = pd.DataFrame(reach_rows)
print(f'(procedure, anatomy, ds_id) addressed tuples: {len(raw):,}')

(procedure, anatomy, ds_id) addressed tuples: 23


In [10]:
# Aggregate to one row per (procedure, anatomy)
def _classify(df):
    has_both_pinned = ((df['proc_state'] == 'pinned') & (df['anat_state'] == 'pinned')).any()
    if has_both_pinned:
        return 'right_grain'
    has_proc_pin = (df['proc_state'] == 'pinned').any()
    has_proc_vlist = (df['proc_state'] == 'vlist').any()
    if has_proc_pin:
        return 'modality_only_anatomy_permitted'
    if has_proc_vlist:
        return 'procedure_in_vlist'
    return 'unclassified'

agg_rows = []
for (pid, psub, aid, asub), g in raw.groupby(
        ['procedure_concept_id', 'procedure_submission_value', 'anatomy_concept_id', 'anatomy_submission_value']):
    pinned_both = g[(g['proc_state'] == 'pinned') & (g['anat_state'] == 'pinned')]
    pin_proc_vlist_anat = g[(g['proc_state'] == 'pinned') & (g['anat_state'] == 'vlist')]
    vlist_proc_pin_anat = g[(g['proc_state'] == 'vlist')  & (g['anat_state'] == 'pinned')]
    vlist_proc_vlist_anat = g[(g['proc_state'] == 'vlist') & (g['anat_state'] == 'vlist')]
    agg_rows.append({
        'procedure_concept_id': pid,
        'procedure_submission_value': psub,
        'anatomy_concept_id': aid,
        'anatomy_submission_value': asub,
        'pr_dss_pins_both': '; '.join(sorted(pinned_both['ds_id'])),
        'pr_dss_pins_proc_anatomy_in_vlist': '; '.join(sorted(pin_proc_vlist_anat['ds_id'])),
        'pr_dss_proc_in_vlist_anatomy_pinned': '; '.join(sorted(vlist_proc_pin_anat['ds_id'])),
        'pr_dss_both_in_value_list': '; '.join(sorted(vlist_proc_vlist_anat['ds_id'])),
        'gap_classification': _classify(g),
    })

pr_reach_out = pd.DataFrame(agg_rows).sort_values(
    ['procedure_submission_value', 'anatomy_submission_value']
).reset_index(drop=True)
print(f'PR_Reachability: {len(pr_reach_out):,} rows x {len(pr_reach_out.columns)} cols')
print()
print('Gap classification distribution:')
print(pr_reach_out['gap_classification'].value_counts())

PR_Reachability: 20 rows x 9 cols

Gap classification distribution:
gap_classification
modality_only_anatomy_permitted    13
procedure_in_vlist                  4
right_grain                         3
Name: count, dtype: int64


## 7. Write workbook

Three sheets: `ReadMe`, `PR_Procedures`, `PR_Reachability`. Color convention follows the repo standard: grey for keys/counts/lists, yellow for COSMoS-derived submission values.

In [11]:
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
WRAP = Alignment(wrap_text=True, vertical='top')

GREEN_HEADER = PatternFill('solid', fgColor='548235')   # TESTCD / SDTM CT side
YELLOW_HEADER = PatternFill('solid', fgColor='FFD700')  # COSMoS side
GREY_HEADER = PatternFill('solid', fgColor='808080')    # keys, aggregation


def write_sheet(ws, df, header_fills, col_widths):
    cols = list(df.columns)
    for ci, name in enumerate(cols, 1):
        cell = ws.cell(row=1, column=ci, value=name)
        cell.font = HEADER_FONT
        cell.fill = header_fills.get(name, GREY_HEADER)
        cell.alignment = WRAP
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        for ci, name in enumerate(cols, 1):
            val = row[name]
            cell = ws.cell(row=ri, column=ci, value=val if val != '' else None)
            cell.font = DATA_FONT
            cell.alignment = WRAP
    for ci, name in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(ci)].width = col_widths.get(name, 18)
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = f'A1:{get_column_letter(len(cols))}1'


print('Writer ready.')

Writer ready.


In [12]:
wb = Workbook()
ws_rm = wb.active
ws_rm.title = 'ReadMe'

readme_font = Font(name='Arial', size=10)
title_font = Font(name='Arial', size=12, bold=True)
section_font = Font(name='Arial', size=10, bold=True)

readme_lines = [
    ('PR_DSS_Reachability — procedure-forward reachability over cosmos-graph', title_font),
    ('', None),
    ('PROVENANCE', section_font),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', readme_font),
    (f'Notebook: consumer-bases/notebooks/30_pr_dss_reachability.ipynb', readme_font),
    (f'Inputs:', readme_font),
    (f'  cosmos-graph/interim/COSMoS_Graph.xlsx (DSS, Variables)', readme_font),
    (f'  cosmos-graph/interim/COSMoS_Graph_CT.xlsx (CodelistTerms)', readme_font),
    ('', None),
    ('SCOPE', section_font),
    ('PR-domain Dataset Specializations and the procedure-anatomy pairs', readme_font),
    ('they address. PROCEDUR codelist C101858 (152 terms); LOC codelist', readme_font),
    ('C74456 (1474 terms). At 2026-Q1 there are 8 PR DSSs.', readme_font),
    ('', None),
    ('SCOPE DISCIPLINE', section_font),
    ('All classifications are mechanical (deterministic from Variables', readme_font),
    ('state). No editorial judgement. consumer-bases scope rule.', readme_font),
    ('', None),
    ('SHEETS', section_font),
    ('PR_Procedures — one row per governed PROCEDUR term (152 rows).', readme_font),
    ('  procedure_concept_id, procedure_submission_value      — identity', readme_font),
    ('  dss_count_pinning                                     — DSSs pinning PRDECOD to this term', readme_font),
    ('  dss_count_in_value_list                               — DSSs with this term in PRDECOD value_list', readme_font),
    ('  dss_count_at_right_grain                              — pinning DSSs that also pin PRLOC', readme_font),
    ('  dss_count_modality_only_anatomy_vlist                 — pinning DSSs with PRLOC value_list', readme_font),
    ('  dss_count_modality_only_anatomy_bare                  — pinning DSSs with PRLOC bare', readme_font),
    ('  dss_ids_pinning, dss_ids_in_value_list                — semicolon-joined ds_id lists', readme_font),
    ('  coverage_classification                               — right_grain_exists / modality_only / no_pr_dss', readme_font),
    ('PR_Reachability — one row per (PROCEDUR, LOC) addressed by some PR DSS.', readme_font),
    ('  procedure / anatomy concept_id and submission_value   — identity', readme_font),
    ('  pr_dss_pins_both                                      — DSSs pinning both', readme_font),
    ('  pr_dss_pins_proc_anatomy_in_vlist                     — DSSs pinning procedure, listing anatomy', readme_font),
    ('  pr_dss_proc_in_vlist_anatomy_pinned                   — RADTHERAPHYBREASTCANCER pattern', readme_font),
    ('  pr_dss_both_in_value_list                             — DSSs ranging on both axes', readme_font),
    ('  gap_classification                                    — right_grain / modality_only_anatomy_permitted /', readme_font),
    ('                                                          procedure_in_vlist', readme_font),
    ('', None),
    ('UNIVERSE — PR_Reachability', section_font),
    ('Only (procedure, anatomy) combinations addressed by at least one', readme_font),
    ('PR DSS appear here. Combinations not addressed are absent — the', readme_font),
    ('gap is the absence of a row, plus the no_pr_dss rows in PR_Procedures.', readme_font),
    ('Full cartesian (152 procedures x 1474 anatomies = 224,048) is not', readme_font),
    ('materialised. Excluded combinations can be derived as', readme_font),
    ('  (touched_procedures x touched_anatomies) MINUS this sheet', readme_font),
    ('if needed by a downstream consumer.', readme_font),
    ('', None),
    ('STATUS', section_font),
    ('Procedure-forward projection counterpart to sdtm-findings-graph.', readme_font),
    ('Sources: COSMoS public exports + NCI EVS CT package 2026-03-27.', readme_font),
]

for ri, (text, font) in enumerate(readme_lines, 1):
    cell = ws_rm.cell(row=ri, column=1, value=text if text else None)
    if font:
        cell.font = font

ws_rm.column_dimensions['A'].width = 100
print(f'ReadMe: {len(readme_lines)} lines')

ReadMe: 49 lines


In [13]:
# ── PR_Procedures sheet ──
ws_p = wb.create_sheet('PR_Procedures')

P_FILLS = {
    'procedure_concept_id':                  GREY_HEADER,
    'procedure_submission_value':            YELLOW_HEADER,
    'dss_count_pinning':                     GREY_HEADER,
    'dss_count_in_value_list':               GREY_HEADER,
    'dss_count_at_right_grain':              GREY_HEADER,
    'dss_count_modality_only_anatomy_vlist': GREY_HEADER,
    'dss_count_modality_only_anatomy_bare':  GREY_HEADER,
    'dss_ids_pinning':                       GREY_HEADER,
    'dss_ids_in_value_list':                 GREY_HEADER,
    'coverage_classification':               GREY_HEADER,
}

P_WIDTHS = {
    'procedure_concept_id':                  14,
    'procedure_submission_value':            38,
    'dss_count_pinning':                     14,
    'dss_count_in_value_list':               16,
    'dss_count_at_right_grain':              16,
    'dss_count_modality_only_anatomy_vlist': 22,
    'dss_count_modality_only_anatomy_bare':  22,
    'dss_ids_pinning':                       30,
    'dss_ids_in_value_list':                 30,
    'coverage_classification':               24,
}

write_sheet(ws_p, pr_procedures_out, P_FILLS, P_WIDTHS)
print(f'PR_Procedures: {len(pr_procedures_out):,} rows x {len(pr_procedures_out.columns)} cols')

PR_Procedures: 152 rows x 10 cols


In [14]:
# ── PR_Reachability sheet ──
ws_r = wb.create_sheet('PR_Reachability')

R_FILLS = {
    'procedure_concept_id':                GREY_HEADER,
    'procedure_submission_value':          YELLOW_HEADER,
    'anatomy_concept_id':                  GREY_HEADER,
    'anatomy_submission_value':            YELLOW_HEADER,
    'pr_dss_pins_both':                    GREY_HEADER,
    'pr_dss_pins_proc_anatomy_in_vlist':   GREY_HEADER,
    'pr_dss_proc_in_vlist_anatomy_pinned': GREY_HEADER,
    'pr_dss_both_in_value_list':           GREY_HEADER,
    'gap_classification':                  GREY_HEADER,
}

R_WIDTHS = {
    'procedure_concept_id':                14,
    'procedure_submission_value':          30,
    'anatomy_concept_id':                  14,
    'anatomy_submission_value':            30,
    'pr_dss_pins_both':                    24,
    'pr_dss_pins_proc_anatomy_in_vlist':   30,
    'pr_dss_proc_in_vlist_anatomy_pinned': 30,
    'pr_dss_both_in_value_list':           24,
    'gap_classification':                  30,
}

write_sheet(ws_r, pr_reach_out, R_FILLS, R_WIDTHS)
print(f'PR_Reachability: {len(pr_reach_out):,} rows x {len(pr_reach_out.columns)} cols')

PR_Reachability: 20 rows x 9 cols


In [15]:
wb.save(OUTPUT_FILE)
print(f'\nWritten: {OUTPUT_FILE}')
print(f'File size: {OUTPUT_FILE.stat().st_size / 1024:.0f} KB')


Written: /sessions/zealous-wonderful-einstein/mnt/cdisc-for-ai/consumer-bases/interim/PR_DSS_Reachability.xlsx
File size: 16 KB


## 8. Summary

In [16]:
print('=== PR_DSS_Reachability summary ===')
print(f'Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')
print()
print(f'PR_Procedures:    {len(pr_procedures_out):>4} rows x {len(pr_procedures_out.columns):>2} cols')
print(f'PR_Reachability:  {len(pr_reach_out):>4} rows x {len(pr_reach_out.columns):>2} cols')
print()
print('PROCEDUR coverage:')
for cls, n in pr_procedures_out['coverage_classification'].value_counts().items():
    print(f'  {cls:30s} {n:>4}  / {len(pr_procedures_out)}')
print()
print('Reachability gap classification:')
for cls, n in pr_reach_out['gap_classification'].value_counts().items():
    print(f'  {cls:35s} {n:>4}')
print()
print('Right-grain PR DSSs (procedure pinned + anatomy pinned):')
rg = pr_reach_out[pr_reach_out['gap_classification'] == 'right_grain']
print(rg[['procedure_submission_value', 'anatomy_submission_value', 'pr_dss_pins_both']].to_string(index=False))

=== PR_DSS_Reachability summary ===
Output: consumer-bases/interim/PR_DSS_Reachability.xlsx

PR_Procedures:     152 rows x 10 cols
PR_Reachability:    20 rows x  9 cols

PROCEDUR coverage:
  no_pr_dss                       149  / 152
  right_grain_exists                3  / 152

Reachability gap classification:
  modality_only_anatomy_permitted       13
  procedure_in_vlist                     4
  right_grain                            3

Right-grain PR DSSs (procedure pinned + anatomy pinned):
procedure_submission_value anatomy_submission_value pr_dss_pins_both
                   CT SCAN                    CHEST      CTSCANCHEST
                       MRI                    BRAIN         MRIBRAIN
                     X-RAY                    CHEST        XRAYCHEST
